# Prequisites

Run every cell in this notebook before starting the course. It checks that your
environment has all api keys correctly set. We will be re-entering api keys in the 5 course notebooks, so we want to make sure it's working here.

## Test API Keys for Extending Exercises with cloud hosted services

### Set API Keys (Optional)

In [ ]:
import os
import getpass
from dotenv import find_dotenv, load_dotenv

# Search from the kernel's working directory up to the repository root.
env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)

# Prompt only when a key was not provided through .env or the shell.
key_prompts = {
    "OPENAI_API_KEY": "OpenAI",
    "LANGSMITH_API_KEY": "Langsmith",
    "PINECONE_API_KEY": "Pinecone",
    "CO_API_KEY": "Cohere",
}

for name, provider in key_prompts.items():
    if not os.getenv(name):
        value = getpass.getpass(f"Enter your {provider} API Key (leave empty to skip): ")
        if value:
            os.environ[name] = value

print(f"API keys loaded from {env_path}" if env_path else "API key setup complete.")

### Test OpenAI API Key

In [ ]:
import os
import openai

if os.getenv("OPENAI_API_KEY"):
    try:
        client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        response = client.models.list()
        print("OpenAI API key is valid! You have access to the following models:")
        for model in response.data[:5]: # Print first 5 models as an example
            print(f"- {model.id}")
    except openai.AuthenticationError:
        print("OpenAI API key is invalid or expired. Please check your key.")
    except Exception as e:
        print(f"An error occurred while testing OpenAI API: {e}")
else:
    print("OpenAI API key not set. Skipping test.")

### Test Langsmith API Key

In [ ]:
import os
from langsmith import Client

if os.getenv("LANGSMITH_API_KEY"):
    try:
        # Initializing the client implicitly tests the API key
        client = Client(api_url="https://api.smith.langchain.com", api_key=os.getenv("LANGSMITH_API_KEY"))
        # A simple operation to confirm connectivity (e.g., listing projects)
        _ = client.list_projects()
        print("Langsmith API key is valid and connected!")
    except Exception as e:
        print(f"Langsmith API key is invalid or an error occurred: {e}")
else:
    print("Langsmith API key not set. Skipping test.")

### Test Pinecone API Key

In [ ]:
!pip install pinecone --quiet

In [ ]:

import os
from pinecone import Pinecone, ServerlessSpec

if os.getenv("PINECONE_API_KEY"):
    try:
        # Ensure to set the environment variable for your Pinecone API key
        # Pinecone client initialization implicitly uses the API key from env
        pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

        # Try listing collections or indexes to confirm connection
        if pc.list_collections():
            print("Pinecone API key is valid and connected. Found existing collections.")
        elif pc.list_indexes():
            print("Pinecone API key is valid and connected. Found existing indexes.")
        else:
            print("Pinecone API key is valid and connected, but no collections or indexes found.")
    except Exception as e:
        print(f"Pinecone API key is invalid or an error occurred: {e}")
else:
    print("Pinecone API key not set. Skipping test.")

### Test Cohere API Key

In [ ]:
!pip install cohere --quiet

In [ ]:
import os
import cohere

key = os.getenv("CO_API_KEY")

if not key:
    raise RuntimeError("CO_API_KEY is not set")

try:
    co = cohere.Client(api_key=key)

    # Lightweight authenticated request; does not run inference
    response = co.models.list(page_size=1)

    print("✅ Cohere API key is valid and the API is reachable")

    if response.models:
        print("Accessible model:", response.models[0].name)

except Exception as exc:
    print("❌ Cohere connection failed")
    print(f"{type(exc).__name__}: {exc}")